# Cryptography (CC4017) -- Week 3

## Chalenge 1

Use Python to encrypt a file in CBC mode and decrypt it. Check for success

In [4]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding


B = int(input("Choose a block size: "))


key = os.urandom(32)  
iv = os.urandom(B)


cipher = Cipher(algorithms.AES(key), modes.CBC(iv))

encryptor = cipher.encryptor()

input_file = input("Choose a file: ")

with open(input_file, 'rb') as file:
    message = file.read()

# Apply padding
padder = padding.PKCS7(B * 8).padder()
padded_message = padder.update(message) + padder.finalize()

ciphertext = encryptor.update(padded_message) + encryptor.finalize()

print("Ciphertext:", ciphertext)

decryptor = cipher.decryptor()

decrypted_padded_message = decryptor.update(ciphertext) + decryptor.finalize()

# Remove padding after decryption
unpadder = padding.PKCS7(B * 8).unpadder()
decrypted_message = unpadder.update(decrypted_padded_message) + unpadder.finalize()

print("Decrypted Message:", decrypted_message.decode('utf-8', errors='ignore'))


Ciphertext: b"\x9d\x1bT\x7f\x9e$\xfc\xf3[\rR\x98V\xdd\xaf\xdcr\x9eHT\x84\xa1\x94\xaa\x8d>\xb3eA\xe9'\xa2"
Decrypted Message: abcdefghijklmnopasdwasdwa



## Chalenge 2

Repeat this process with OpenSSL

## Chalenge 3


Edit the file to change the value of (but not delete!) one byte and decrypt again

In [8]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

B = int(input("Choose a block size: "))

key = os.urandom(32)  
iv = os.urandom(B)

cipher = Cipher(algorithms.AES(key), modes.CBC(iv))

encryptor = cipher.encryptor()

input_file = input("Choose a file: ")

with open(input_file, 'rb') as file:
    message = file.read()

# Apply padding
padder = padding.PKCS7(B * 8).padder()
padded_message = padder.update(message) + padder.finalize()

ciphertext = encryptor.update(padded_message) + encryptor.finalize()

print("Ciphertext:", ciphertext)

modified_byte = int(input("Enter a new value for the first byte (0-255): "))
ciphertext = bytes([modified_byte]) + ciphertext[1:]

print("Modified Ciphertext:", ciphertext)

decryptor = cipher.decryptor()

decrypted_padded_message = decryptor.update(ciphertext) + decryptor.finalize()

# Remove padding after decryption
unpadder = padding.PKCS7(B * 8).unpadder()
decrypted_message = unpadder.update(decrypted_padded_message) + unpadder.finalize()

print("Decrypted Message:", decrypted_message.decode('utf-8', errors='ignore'))


Ciphertext: b'\xc9\xd0\xd7\nM\xd3\xfdx;89\xa8\xee\xa6u1?+G\xf8,6\xc4\xb4\xac\xa8\xa3`\t\xb4>\x17'
Modified Ciphertext: b'Z\xd0\xd7\nM\xd3\xfdx;89\xa8\xee\xa6u1?+G\xf8,6\xc4\xb4\xac\xa8\xa3`\t\xb4>\x17'
Decrypted Message: Ӧ&>Z@HN>Xsdwasdwa



### 3.1 What happened?

The information that was encoded was changed, making it so when decrypting, the messaged that shows up is different than the one originally sent for encryption

### 3.2 Could you recover a file encrypted with CBC if the IV and the first ciphertext block were corrupted or lost?

With the IV and the CBC lost or corrupted the whole file would not be recoverable. We could, at max, recover every block up to the second one. The first and second blocks would not be recoverable as the information in the first block and the iv was modified or lost. As such, there's no way to use them to XOR with the decrypted (through the key) block, in order to obtain the original plain text of the block, it would be different than the original

### 3.3 Could you recover it if during a satellite transmission one bit of the ciphertext is not delivered?

The affected block will be altered, resulting in corruption in the output. This also means that the subsequent blocks will also likely have changed bits that are incorrect due to cascading errors. Ultimately, the loss of a bit changes the size of the message which can lead to padding errors, since the 16 byte blocks are not correctly completed (the last one will be missing one bit). This all implies that if, somehow, the loss of a bit does not throw an error due to padding problems, making it impossible to decypher the message, the subsequent information that is cyphered will all be incorrectly decyphered after that one lost bit.

### 3.4 Could you modify a byte in the middle of a CBC encrypted file without fully re-encrypting it?

Due to the nature of CBC, modifying the content of an encrypted file without decrypting, altering the contents and re-encrypting it is very hard. If other modes are used, it could, however, be possible to change the information without having to re-encrypt the whole file.

## Chalenge 4

Repeat the exercise with CTR mode. What are the differences?

In [1]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

B = int(input("Choose a block size: "))

key = os.urandom(32)  
nonce = os.urandom(B)

cipher = Cipher(algorithms.AES(key), modes.CTR(nonce))

encryptor = cipher.encryptor()

input_file = input("Choose a file: ")

with open(input_file, 'rb') as file:
    message = file.read()

ciphertext = encryptor.update(message) + encryptor.finalize()

print("Ciphertext:", ciphertext)

modified_byte = int(input("Enter a new value for the first byte (0-255): "))
ciphertext = bytes([modified_byte]) + ciphertext[1:]

print("Modified Ciphertext:", ciphertext)

decryptor = cipher.decryptor()

decrypted_message= decryptor.update(ciphertext) + decryptor.finalize()

print("Decrypted Message:", decrypted_message.decode('utf-8', errors='ignore'))


Ciphertext: b'\x11\x87[\xaf;\x86)\xb7R\x16\xc3lFQ\xd1,=\xfaTa\x9e\xef\xc0\xcc\x95\x80'
Modified Ciphertext: b'\x0e\x87[\xaf;\x86)\xb7R\x16\xc3lFQ\xd1,=\xfaTa\x9e\xef\xc0\xcc\x95\x80'
Decrypted Message: ~bcdefghijklmnopasdwasdwa



### 4.1 What happened?

Only the first character was affected, seemengly disappearing, meaning that changing the first byte, only altered the first block of bytes

### 4.2 Could you recover a file encrypted with CTR if the nonce and the first ciphertext block were corrupted or lost?

In this case, the nonce, called IV on CBC, is essential, meaning the whole file would be lost, since without it no part of the ciphertext can be decyphered. If only the first block is lost, however, the impact isn't as big as in the CBC mode. In this case, only the first block of plain text would not be recoverable, meaning the rest of the file would.

### 4.3 Could you recover it if during a satellite transmission one bit of the ciphertext is not delivered?

In this case, a shifiting of bits occurs after the position of the missing bit. With that happening the decyphering of all the cypher after the missing bit will be done incorrectly 

### 4.4 Could you modify a byte in the middle of a CTR encrypted file without fully re-encrypting it?

Due to the independent nature of the blocks in CTR encryption, you can change the content of a byte, only having to re-encrypt that said byte. This means that as long as we have the correct information for the re-encryption of the byte, no problem would arise, and there would be no need to decrypt the whole file, specially because changing certain bytes on the plain text will only result on the change of certain bytes on the encrypted text.